In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import hashlib
import hmac as hmac_lib
import secrets
import numpy as np
import matplotlib.pyplot as plt

from ecc import *

---

# Módulo 6: Aplicaciones en Bitcoin

**Dónde estamos:** Construimos ECDSA desde cero — firma, verificación, el
peligro del nonce, y cómo las firmas se convierten en bytes. ECDSA es lo que
Bitcoin usó de 2009 a 2021. Pero tiene peculiaridades que complicaron la vida
de los desarrolladores. Este módulo cubre lo que lo reemplazó y dónde más
aparece la ECC en Bitcoin.

## 6.1 De ECDSA a Schnorr — por qué Bitcoin cambió

ECDSA funciona, pero tiene tres problemas prácticos:

**1. Firmas de longitud variable.** La codificación DER produce 71-73 bytes
dependiendo de si $r$ o $s$ necesitan un byte de relleno.

**2. Maleabilidad.** Dada una firma válida $(r, s)$, cualquiera puede cambiarla
a $(r, N-s)$ — también válida. Esto cambia el hash de la transacción sin
invalidarla.

**3. Sin linealidad.** No se pueden combinar firmas ECDSA. Si Alice y Bob
firman ambos, necesitas dos firmas separadas en la transacción.

Las firmas Schnorr (inventadas por Claus-Peter Schnorr en 1989, patente
expirada en 2008) solucionan las tres:

| Problema | ECDSA | Schnorr |
|---------|-------|---------|
| Tamaño | ~72 bytes (variable, DER) | 64 bytes (fijo) |
| Maleabilidad | Requiere arreglo low-S | Ninguna por diseño |
| Linealidad | No lineal | **Lineal** — las firmas se suman |
| Verificación en lote | Una a la vez | Verificar muchas a la vez |

La linealidad es lo importante. Porque Schnorr es lineal, múltiples firmantes
pueden producir una **sola firma** que se ve idéntica a una firma de un solo
firmante (esto es lo que hace MuSig2).

### Cómo funciona la firma Schnorr

La ecuación es dramáticamente más simple que ECDSA:

```
  Firma:                             Verificación:
  ──────                             ─────────────
  k = nonce aleatorio                e = H(R_x || P_x || m)
  R = k × G                         Verificar: s × G  =  R + e × P
  e = H(R_x || P_x || m)
  s = k + e·d  mod N                Eso es todo. Una ecuación.
  Firma: (R_x, s) = 64 bytes
```

### Por qué funciona la verificación

Misma intuición que ECDSA — la clave privada se cancela:

$$s \times G = (k + ed) \times G = kG + edG = R + eP \quad \checkmark$$

### Por qué importa la linealidad

Si Alice tiene clave $d_A$ y Bob tiene clave $d_B$, su clave combinada es
$P_{AB} = P_A + P_B = (d_A + d_B) \times G$. Cada uno puede producir una
firma parcial y sumarlas: $s = s_A + s_B$. Esa es una firma Schnorr válida
para la clave combinada. Con ECDSA, el $k^{-1}$ en la ecuación destruye esta
linealidad.

Bitcoin activó Schnorr con el soft fork **Taproot** (noviembre 2021, bloque
709,632). Cada gasto key-path P2TR usa Schnorr.

In [ ]:
# Simplified Schnorr signature (BIP340-like)

def tagged_hash(tag: str, data: bytes) -> bytes:
    """BIP340 tagged hash: SHA256(SHA256(tag) || SHA256(tag) || data)."""
    tag_hash = hashlib.sha256(tag.encode()).digest()
    return hashlib.sha256(tag_hash + tag_hash + data).digest()

def schnorr_sign(message: bytes, private_key: int) -> bytes:
    """Simplified BIP340 Schnorr signature."""
    P = scalar_mult(private_key, G)
    d = private_key if P.y % 2 == 0 else SECP_N - private_key
    
    # Deterministic nonce (simplified)
    aux = secrets.token_bytes(32)
    t = int.from_bytes(aux, 'big') ^ d
    k_bytes = tagged_hash("BIP0340/nonce", 
                          t.to_bytes(32, 'big') + P.x.to_bytes(32, 'big') + message)
    k = int.from_bytes(k_bytes, 'big') % SECP_N
    if k == 0:
        raise ValueError("k is zero")
    
    R = scalar_mult(k, G)
    if R.y % 2 != 0:
        k = SECP_N - k
        R = scalar_mult(k, G)
    
    e_bytes = tagged_hash("BIP0340/challenge",
                          R.x.to_bytes(32, 'big') + P.x.to_bytes(32, 'big') + message)
    e = int.from_bytes(e_bytes, 'big') % SECP_N
    
    s = (k + e * d) % SECP_N
    return R.x.to_bytes(32, 'big') + s.to_bytes(32, 'big')

def schnorr_verify(message: bytes, signature: bytes, pubkey_x: int) -> bool:
    """Simplified BIP340 Schnorr verification."""
    R_x = int.from_bytes(signature[:32], 'big')
    s = int.from_bytes(signature[32:], 'big')
    
    # Recover P (assume even y)
    y2 = (pow(pubkey_x, 3, SECP_P) + 7) % SECP_P
    y = mod_sqrt(y2, SECP_P)
    if y % 2 != 0:
        y = SECP_P - y
    P = Point(pubkey_x, y)
    
    e_bytes = tagged_hash("BIP0340/challenge",
                          R_x.to_bytes(32, 'big') + pubkey_x.to_bytes(32, 'big') + message)
    e = int.from_bytes(e_bytes, 'big') % SECP_N
    
    # Verify: s×G = R + e×P
    lhs = scalar_mult(s, G)
    
    # Recover R (assume even y)
    R_y2 = (pow(R_x, 3, SECP_P) + 7) % SECP_P
    R_y = mod_sqrt(R_y2, SECP_P)
    if R_y % 2 != 0:
        R_y = SECP_P - R_y
    R = Point(R_x, R_y)
    
    rhs = point_add(R, scalar_mult(e, P))
    return lhs == rhs

# Demo
d_schnorr = secrets.randbelow(SECP_N - 1) + 1
P_schnorr = scalar_mult(d_schnorr, G)
msg_schnorr = b"Taproot transaction"

sig_schnorr = schnorr_sign(msg_schnorr, d_schnorr)
valid_schnorr = schnorr_verify(msg_schnorr, sig_schnorr, P_schnorr.x)

print("=== Firma Schnorr (BIP340) ===")
print(f"Mensaje: {msg_schnorr.decode()}")
print(f"Firma: {sig_schnorr.hex()[:40]}...")
print(f"  R_x (32 bytes) + s (32 bytes) = {len(sig_schnorr)} bytes total")
print(f"Verificación: {valid_schnorr}  ✓")
print(f"\nCompare: ECDSA ≈72 bytes (DER),  Schnorr = 64 bytes (fixed)")

## 6.2 Firmas en el Cable — ECDSA vs Schnorr

En el Módulo 5 construimos la codificación DER para firmas ECDSA (~72 bytes,
variable). Schnorr es más simple: 64 bytes fijos, solo `R_x (32 bytes) || s (32 bytes)`.

| Formato | Codificación | Tamaño | Maleabilidad |
|---------|------------|--------|-------------|
| ECDSA | DER (variable) | ~71-73 bytes + sighash | Requiere arreglo low-S (BIP 62) |
| Schnorr | Fijo `R_x \|\| s` | 64 bytes | Ninguna por diseño |

### ¿Dónde vive la firma en una transacción?

Una transacción de Bitcoin es un flujo de bytes con este diseño:

```
 Transacción SegWit (BIP 141)
┌──────────┬────────┬──────┬────────┬─────────┬─────────┬──────────┐
│ versión  │ marker │ flag │ inputs │ outputs │ witness │ locktime │
│ 4 bytes  │  0x00  │ 0x01 │  var   │   var   │   var   │ 4 bytes  │
└──────────┴────────┴──────┴────────┴─────────┴─────────┴──────────┘
```

La ubicación de la firma depende del tipo de script:

| Tipo de script | Dónde va la firma | Qué más se incluye |
|-------------|----------------------|----------------------|
| **P2PKH** (legado) | campo `scriptSig` del input | clave pública comprimida (33 bytes) |
| **P2WPKH** (SegWit v0) | campo `witness` (scriptSig vacío) | clave pública comprimida (33 bytes) |
| **P2TR key-path** (Taproot) | campo `witness` — solo la firma de 64 bytes | nada más necesario |
| **P2TR script-path** (Taproot) | campo `witness` — firmas + script + bloque de control | tapscript, clave interna, bit de paridad |

```
 Witness P2TR Key-Path
┌───────────────┬──────────────┬──────────────────┐
│ stack items: 1│ item len: 64 │ firma (64 B)      │
│    0x01       │    0x40      │  R_x || s         │
└───────────────┴──────────────┴──────────────────┘
```

In [ ]:
def der_encode_integer(value: int) -> bytes:
    b = value.to_bytes((value.bit_length() + 7) // 8, 'big')
    if b[0] & 0x80:
        b = b'\x00' + b
    return bytes([0x02, len(b)]) + b

def der_encode_signature(r: int, s: int) -> bytes:
    r_der = der_encode_integer(r)
    s_der = der_encode_integer(s)
    body = r_der + s_der
    return bytes([0x30, len(body)]) + body

r_example = 0xDEADBEEF_CAFEBABE_12345678_9ABCDEF0_DEADBEEF_CAFEBABE_12345678_9ABCDEF0
s_example = 0xFEDCBA98_76543210_FEDCBA98_76543210_FEDCBA98_76543210_FEDCBA98_76543210

print("=== ECDSA: Codificación DER ===")
der_sig = der_encode_signature(r_example, s_example)
der_with_sighash = der_sig + bytes([0x01])
print(f"DER + sighash: {len(der_with_sighash)} bytes (variable)")

print(f"\n=== Schnorr: Formato Fijo ===")
schnorr_sig = r_example.to_bytes(32, 'big') + s_example.to_bytes(32, 'big')
print(f"R_x || s:      {len(schnorr_sig)} bytes (fixed)")

print(f"\n=== Testigo P2TR Key-Path ===")
witness = bytes([0x01, 0x40]) + schnorr_sig
print(f"Witness: {len(witness)} bytes total")
print(f"  0x01 = 1 stack item")
print(f"  0x40 = 64 bytes")
print(f"  {schnorr_sig.hex()[:24]}...{schnorr_sig.hex()[-8:]} = signature")
print(f"\nAhorro por entrada: ~{len(der_with_sighash) - len(schnorr_sig)} bytes"
      f" ({len(der_with_sighash)} → {len(schnorr_sig)})")

## 6.3 ECDH en el Enrutamiento Cebolla de Lightning

### El problema

Los pagos Lightning se enrutan a través de una cadena de nodos: Alice → Bob → Carol →
Dave. Cada nodo reenvía el pago al siguiente. Pero hay un problema de privacidad:
si cada nodo puede ver la ruta completa, entonces Bob sabe que Alice está pagando
a Dave a través de Carol.

La solución es el **enrutamiento cebolla** — la misma idea que Tor. Alice envuelve
las instrucciones de pago en capas de cifrado, una capa por salto. Cada nodo solo
puede pelar su propia capa, que le dice "reenvía esto al siguiente nodo."

### Dónde entra la ECC

Para cifrar cada capa, Alice necesita un secreto compartido con cada salto — pero
no puede hacer un intercambio interactivo Diffie-Hellman con cada nodo. En su
lugar, usa **ECDH** — la versión de curvas elípticas del Diffie-Hellman
unilateral del Módulo 4.

Alice elige un secreto efímero $e$ y calcula un secreto compartido con cada
salto usando su clave pública publicada:

```
  Clave efímera de Alice: e (secreto), E = e×G (público, enviado con el pago)

  Para salto 1 (Bob, clave pública P₁):
    Secreto compartido:  S₁ = SHA256(e × P₁)
    Bob puede calcular:  S₁ = SHA256(d₁ × E)  ← mismo resultado, ECDH

  Para salto 2 (Carol, clave pública P₂):
    Secreto compartido:  S₂ = SHA256(e × P₂)
    Carol puede calcular: S₂ = SHA256(d₂ × E')  ← E' está cegada (ver abajo)

  Para salto 3 (Dave, clave pública P₃):
    Secreto compartido:  S₃ = SHA256(e × P₃)
    Dave puede calcular:  S₃ = SHA256(d₃ × E'')
```

¿Por qué $e \times P_i = d_i \times E$? Misma razón que Diffie-Hellman funciona —
conmutatividad:
$$e \times P_i = e \times (d_i \times G) = d_i \times (e \times G) = d_i \times E$$

### El truco del cegamiento

Si cada salto ve la misma clave efímera $E$, Bob y Carol podrían coludirse y
darse cuenta de que están en la misma ruta. Así que después de cada salto, la
clave efímera se **ciega** — se multiplica por un factor derivado del secreto
compartido:

$$E' = E \times \text{SHA256}(E \| S_i)$$

Cada salto ve una versión diferente de $E$. Ningún par de saltos puede vincular
sus vistas.

### Del secreto compartido al cifrado

Cada secreto compartido $S_i$ se usa para derivar claves de cifrado reales:
- clave `rho` → cifra info de enrutamiento (ChaCha20)
- clave `mu` → autentica el payload (HMAC)

Alice construye la cebolla cifrando de adentro hacia afuera: la capa de Dave
primero, luego la de Carol, luego la de Bob.

In [ ]:
import hmac as hmac_lib

def ecdh_shared_secret(my_private: int, their_public: Point) -> bytes:
    """ECDH: compute shared secret from private key and other party's public key."""
    shared_point = scalar_mult(my_private, their_public)
    compressed = serialize_compressed(shared_point)
    return hashlib.sha256(compressed).digest()

def generate_key(shared_secret: bytes, key_type: str) -> bytes:
    """Derive a specific key from shared secret (BOLT #4 key derivation)."""
    return hmac_lib.new(key_type.encode(), shared_secret, hashlib.sha256).digest()

def blind_ephemeral(ephemeral_pub: Point, shared_secret: bytes) -> Point:
    """Blind ephemeral key so next hop can't link it to previous hop."""
    pub_bytes = serialize_compressed(ephemeral_pub)
    blind_bytes = hashlib.sha256(pub_bytes + shared_secret).digest()
    blind_factor = int.from_bytes(blind_bytes, 'big') % SECP_N
    return scalar_mult(blind_factor, ephemeral_pub)

# Simulate 3-hop onion routing
print("=== Enrutamiento Cebolla Lightning (ECDH) ===")
print("Sender → Hop1 → Hop2 → Hop3 (recipient)")

# Each hop has a key pair
hops = []
for i in range(3):
    d_hop = secrets.randbelow(SECP_N - 1) + 1
    P_hop = scalar_mult(d_hop, G)
    hops.append({'private': d_hop, 'public': P_hop, 'name': f'Hop{i+1}'})

# Sender creates ephemeral key pair
e_priv = secrets.randbelow(SECP_N - 1) + 1
E = scalar_mult(e_priv, G)  # Ephemeral public key (sent with packet)
print(f"\nE efímera del remitente: {serialize_compressed(E).hex()[:20]}...")

# Sender computes all shared secrets
ephemeral = E
sender_secrets = []
for hop in hops:
    ss = ecdh_shared_secret(e_priv, hop['public'])
    rho = generate_key(ss, "rho")
    mu = generate_key(ss, "mu")
    sender_secrets.append({'ss': ss, 'rho': rho, 'mu': mu})
    
    # Blind for next hop
    blind_bytes = hashlib.sha256(serialize_compressed(ephemeral) + ss).digest()
    blind_factor = int.from_bytes(blind_bytes, 'big') % SECP_N
    e_priv = (e_priv * blind_factor) % SECP_N
    ephemeral = scalar_mult(e_priv, G)

# Each hop computes the same shared secret using its private key
E_current = E
for i, hop in enumerate(hops):
    hop_ss = ecdh_shared_secret(hop['private'], E_current)
    match = hop_ss == sender_secrets[i]['ss']
    print(f"\n{hop['name']}:")
    print(f"  Recibe E: {serialize_compressed(E_current).hex()[:20]}...")
    print(f"  Calcula: d×E = secreto compartido")
    print(f"  El secreto compartido coincide con el del remitente: {match}  ✓")
    print(f"  Deriva: rho (cifrar), mu (HMAC)")
    
    # Blind E for next hop
    blind_bytes = hashlib.sha256(serialize_compressed(E_current) + hop_ss).digest()
    blind_factor = int.from_bytes(blind_bytes, 'big') % SECP_N
    E_current = scalar_mult(blind_factor, E_current)

print(f"\nPunto clave: cada salto ve una E diferente (el cegamiento impide vincularlos).")
print(f"Nadie excepto el remitente conoce la ruta completa.")

---